# Solutions · Chapter 05-02 · Simple linear regression by hand

Worked answers for `notebooks/05_regression/05-02_simple_linear.ipynb`.

**E9, E10 and E15 are the three worth reading**: one shows least squares losing its nerve, one shows that
"the line" is two different lines depending on which way you fit it, and one is a bug you can diagnose from
a single number.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# TINY, SYNTHETIC: the chapter's nine deliveries
distance = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
minutes = np.array([16, 20, 25, 25, 30, 31, 36, 42, 45], dtype=float)


def fit_line(x, y):
    """The closed form, with no library call."""
    x_bar, y_bar = x.mean(), y.mean()
    slope = ((x - x_bar) * (y - y_bar)).sum() / ((x - x_bar) ** 2).sum()
    return slope, y_bar - slope * x_bar


SLOPE, INTERCEPT = fit_line(distance, minutes)
print("the chapter's line: minutes = %.1f + %.1f x distance" % (INTERCEPT, SLOPE))

## Quick understanding

### E1

`slope = sum((x - x̄)(y - ȳ)) / sum((x - x̄)²)` and `intercept = ȳ - slope × x̄`.

**The numerator** measures how the two columns move *together*: each point contributes the product of its
two deviations, positive when it is on the same side of both means and negative when it breaks the
pattern. **The denominator** measures how much `x` varies on its own, and its job is to convert the
numerator into units of *y per unit of x*.

### E2

Because the residuals must sum to zero. If they summed to anything else, shifting the whole line up or
down by the average residual would reduce the squared error - so at the optimum `ȳ = intercept + slope ×
x̄`, which is exactly the statement that the line passes through `(x̄, ȳ)`.

### E3

`R² = 1 - (sum of squared residuals) / (sum of squared deviations from the mean)`, which is the **skill
score under squared error against the mean baseline**.

**Negative means the model is worse than predicting the mean** - which is impossible for a least-squares
fit on the data it was fitted to, and entirely possible on held-out data.

## Hand calculation

### E4

`x = 1, 2, 3, 4, 5` and `y = 3, 5, 4, 8, 10`. **x̄ = 3, ȳ = 6.**

| x | y | x - x̄ | y - ȳ | product | (x - x̄)² |
|---|---|---|---|---|---|
| 1 | 3 | -2 | -3 | 6 | 4 |
| 2 | 5 | -1 | -1 | 1 | 1 |
| 3 | 4 | 0 | -2 | 0 | 0 |
| 4 | 8 | 1 | 2 | 2 | 1 |
| 5 | 10 | 2 | 4 | 8 | 4 |
| | | | **sums** | **17** | **10** |

**slope = 17 / 10 = 1.7**, and **intercept = 6 - 1.7 × 3 = 0.9**.

### E5

Predictions: `0.9 + 1.7x` gives 2.6, 4.3, 6.0, 7.7, 9.4.

Residuals: **+0.4, +0.7, -2.0, +0.3, +0.6**, which sum to **0.0** exactly.

### E6

The shift formula: shifting a line up by `s` adds `n × s²` to the sum of squared errors. Here `9 × 2²` =
**36**, so the new total is `17 + 36` = **53**.

### E7

`3.5 minutes per kilometre` = `3.5 / 1000` = **0.0035 minutes per metre.**

The **intercept is unchanged at 12.5 minutes**, because it is the prediction at zero - and zero
kilometres is the same place as zero metres.

**Only the slope changes, and it changes because it carries units.** This is the clearest possible
demonstration that the size of a coefficient says nothing on its own: the relationship is identical, the
fit is identical, and the number differs by a factor of a thousand.

In [ ]:
small_x = np.array([1, 2, 3, 4, 5], dtype=float)
small_y = np.array([3, 5, 4, 8, 10], dtype=float)
slope_small, intercept_small = fit_line(small_x, small_y)
residuals_small = small_y - (intercept_small + slope_small * small_x)

print("E4  slope %.1f, intercept %.1f" % (slope_small, intercept_small))
print("E5  residuals %s, summing to %.10f"
      % (np.round(residuals_small, 2), round(float(residuals_small.sum()), 10) + 0.0))
print()
shifted = (minutes - (INTERCEPT + 2 + SLOPE * distance)) ** 2
print("E6  formula: 17 + 9 x 2^2 = %d      directly: %.1f" % (17 + 9 * 4, shifted.sum()))
print()
print("E7  in metres: slope %.5f minutes per metre, intercept %.1f minutes (unchanged)"
      % (SLOPE / 1000, INTERCEPT))

## Coding

### E8 - the closed form

In [ ]:
check_rng = np.random.default_rng(0)
random_x = check_rng.normal(size=200)
random_y = 4 - 2.3 * random_x + check_rng.normal(0, 1.5, 200)

for label, x, y in [("the nine deliveries", distance, minutes), ("200 random points", random_x, random_y)]:
    mine = fit_line(x, y)
    theirs = np.polyfit(x, y, 1)
    print("%-20s  mine %9.6f %9.6f   polyfit %9.6f %9.6f"
          % (label, mine[0], mine[1], theirs[0], theirs[1]))

### E9 - one delivery goes wrong

In [ ]:
damaged = minutes.copy()
damaged[-1] = 95.0

squared_fit = fit_line(distance, damaged)

slope_grid = np.linspace(1.0, 9.0, 801)
intercept_grid = np.linspace(-5.0, 30.0, 801)
absolute_fit = min(((float(np.abs(damaged - (b + w * distance)).mean()), w, b)
                    for b in intercept_grid for w in slope_grid))[1:]

print("%-34s %8s %10s" % ("", "slope", "intercept"))
print("%-34s %8.3f %10.3f" % ("original data, least squares", SLOPE, INTERCEPT))
print("%-34s %8.3f %10.3f" % ("with the outlier, least squares", squared_fit[0], squared_fit[1]))
print("%-34s %8.3f %10.3f" % ("with the outlier, absolute error", absolute_fit[0], absolute_fit[1]))

fig, ax = plt.subplots(figsize=(8.5, 5))
grid = np.linspace(0, 10, 2)
ax.scatter(distance[:-1], damaged[:-1], s=90, color="#0072B2", zorder=3)
ax.scatter([distance[-1]], [damaged[-1]], s=150, marker="X", color="#D55E00", zorder=4,
           label="the ruined delivery")
ax.plot(grid, INTERCEPT + SLOPE * grid, color="#999999", linestyle=":", linewidth=2,
        label="original fit (3.50)")
ax.plot(grid, squared_fit[1] + squared_fit[0] * grid, color="#D55E00", linewidth=2.4,
        label="least squares now (%.2f)" % squared_fit[0])
ax.plot(grid, absolute_fit[1] + absolute_fit[0] * grid, color="#009E73", linewidth=2.4,
        linestyle="--", label="absolute error now (%.2f)" % absolute_fit[0])
ax.set_xlabel("distance (km)")
ax.set_ylabel("minutes")
ax.set_xlim(0, 10)
ax.set_title("One bad point nearly doubles the least-squares slope", fontsize=11.5)
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

**Least squares goes from 3.50 to 6.83 - nearly double - and the absolute-error line does not move at
all.** It stays at exactly 3.500 and 12.500.

That is a stronger result than 05-01's, and worth pausing on. In 05-01 the outlier moved the mean while the
median moved by one minute. Here the robust fit moves by **nothing measurable**, because the ruined point
was already the furthest to the right and above the line; making it further above does not change which
side of the line it is on, and absolute error only counts sides.

**Which would I report? The absolute-error line, and the fact that one point changed the other by 95%.**
Then I would go and find out what happened on that delivery - because a fit that is this sensitive to one
observation is telling you the observation matters more than the model does.

### E10 - fitting it the other way round

In [ ]:
forward_slope, forward_intercept = fit_line(distance, minutes)
reverse_slope, reverse_intercept = fit_line(minutes, distance)

# rewrite "distance = a + b x minutes" as a line in the same axes
implied_slope = 1 / reverse_slope
implied_intercept = -reverse_intercept / reverse_slope

fig, ax = plt.subplots(figsize=(8, 5))
grid = np.linspace(0, 10, 2)
ax.scatter(distance, minutes, s=90, color="#0072B2", zorder=3)
ax.plot(grid, forward_intercept + forward_slope * grid, color="#D55E00", linewidth=2.4,
        label="minutes on distance (slope %.4f)" % forward_slope)
ax.plot(grid, implied_intercept + implied_slope * grid, color="#009E73", linewidth=2.4,
        linestyle="--", label="distance on minutes (implies %.4f)" % implied_slope)
ax.plot([distance.mean()], [minutes.mean()], "o", color="#000000", markersize=10)
ax.annotate("both pass through\n(x̄, ȳ)", (distance.mean(), minutes.mean()),
            textcoords="offset points", xytext=(12, -34), fontsize=9)
ax.set_xlabel("distance (km)")
ax.set_ylabel("minutes")
ax.set_xlim(0, 10)
ax.set_title("Two regressions, two lines, one dataset", fontsize=11.5)
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

print("forward slope (minutes per km)            : %.4f" % forward_slope)
print("reverse fit implies (minutes per km)      : %.4f" % implied_slope)
print("product of the two fitted slopes          : %.6f" % (forward_slope * reverse_slope))
print("the squared correlation                   : %.6f" % np.corrcoef(distance, minutes)[0, 1] ** 2)

**No, they are not the same line** - 3.5000 against an implied 3.5810 - and they cross at `(x̄, ȳ)`,
which both are obliged to pass through.

The identity that explains it:

> **(slope of y on x) × (slope of x on y) = r²**

Here `3.5000 × 0.27926 = 0.9774`, which is exactly the squared correlation.

The reason is `slope = r × (sy/sx)`. Fitting the other way gives `r × (sx/sy)`, and multiplying them
cancels the spreads and leaves `r²`. **The two lines coincide only when `r² = 1`**, that is, when the
points lie exactly on a line; the weaker the relationship, the further apart they are.

**The asymmetry is not a defect.** Least squares minimises *vertical* distances, so "predict y from x" and
"predict x from y" are genuinely different questions with different answers. Which one you want is decided
by which variable you will actually have at prediction time - 04-01's availability question, arriving in
the middle of some algebra.

### E11 - which delivery is holding up the line?

In [ ]:
leave_one_out = []
for dropped in range(len(distance)):
    keep = np.ones(len(distance), dtype=bool)
    keep[dropped] = False
    slope_without, _ = fit_line(distance[keep], minutes[keep])
    leave_one_out.append({"dropped": dropped + 1,
                          "its distance": int(distance[dropped]),
                          "slope without it": round(slope_without, 4),
                          "change": round(slope_without - SLOPE, 4)})
influence = pd.DataFrame(leave_one_out)
print(influence.to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.bar(influence.dropped, influence["slope without it"], color="#0072B2", width=0.6)
ax.axhline(SLOPE, color="#D55E00", linewidth=2, label="slope with all nine (%.2f)" % SLOPE)
ax.set_ylim(3.3, 3.65)
ax.set_xlabel("which delivery was left out")
ax.set_ylabel("refitted slope")
ax.set_title("Nine fits, each missing one delivery", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The slope ranges from 3.3929 to 3.5811 - it moves by up to 3% when any single delivery is removed.**

The two that matter most are **8 and 9, the furthest away**, and that is not a coincidence: a point's
influence on the slope grows with its distance from `x̄`, because the slope formula weights each residual
by `(x - x̄)`.

**Two deliveries have exactly zero influence, for two different reasons**, and both are worth reading:

- **Delivery 5, at 5 km**, sits exactly at `x̄`. Its weight `(x - x̄)` is zero, so it contributes nothing
  to the slope no matter how far off the line it is.
- **Delivery 1, at 1 km**, has a weight of -4 - the largest of any point - but its residual is exactly
  **0**. It lies precisely on the fitted line, so it is not pulling in either direction.

**Influence needs both**: a point far from `x̄` *and* off the line. That product is what "influence" means
in regression, and it is why the dangerous observations are the ones at the extremes of `x` that also
misbehave - not the ones with the largest residual, and not the ones furthest out.

**The general lesson for a nine-point dataset: any conclusion is one observation away from a different
conclusion.** This is the same instinct 04-03 built about splits, applied to rows: before believing a
slope, find out how much of it rests on the few points at the ends.

### E12 - the two normal equations

In [ ]:
rows = []
for label, b, w in [("the fitted line", INTERCEPT, SLOPE),
                    ("through (x̄, ȳ), wrong slope", 17.5, 2.5),
                    ("right slope, shifted up", INTERCEPT + 3, SLOPE),
                    ("wrong in both ways", 8.0, 4.0)]:
    leftover = minutes - (b + w * distance)
    rows.append({"line": label, "intercept": b, "slope": w,
                 "sum(residual)": round(float(leftover.sum()), 6),
                 "sum(residual x distance)": round(float((leftover * distance).sum()), 6)})
print(pd.DataFrame(rows).to_string(index=False))

**Both quantities are zero for the fitted line and for nothing else**, which is what "the normal
equations" means: two conditions, two unknowns, one solution.

The second row is the instructive one. That line has the **wrong slope** and yet its residuals sum to
exactly zero - because it still passes through `(x̄, ȳ)`. So the first condition is not a test of a good
fit at all; **every line through the centre of the data passes it**, of which there are infinitely many.
What that line fails is the second condition, at 60 rather than 0: there is still a tilt in the leftovers.

So the two equations are not interchangeable checks of the same thing:

- **`sum(residual) = 0`** says *the line is at the right height*. It fixes the intercept given a slope.
- **`sum(residual × x) = 0`** says *the line is at the right angle*. It fixes the slope.

The third and fourth rows fail both, which is the ordinary case. **Only the intersection of the two
conditions is a single line**, and that is why "residuals sum to zero" is worth nothing as evidence on its
own - a point the chapter made and this table demonstrates.

## Interpretation

### E13

**"Against what baseline, and measured on which rows?"**

R-squared is the skill score against **the mean**, so 0.98 says the line removed 98% of the error a
constant would have made. That is only impressive if the constant was a serious competitor. On data with
a strong trend the mean is a straw man - and 04-02 showed a case where the right baseline (a per-entity
constant) beat two real models, which would have made every R-squared quoted against the mean look
absurdly good.

**"Is that on the data it was fitted to?"** On nine points fitted with two parameters, R-squared is
guaranteed to look good; it cannot go down when you add a feature. The number that matters is on rows the
fit has not seen.

A third, if there is time: **what is the error in minutes?** 0.98 is not something a dispatcher can act
on; 1.37 minutes of average error is.

### E14

The line predicts `12.5 + 3.5 × 6` = **33.5 minutes** for a 6 km delivery.

**That is not what to promise**, for two reasons this course has already established:

**The line predicts the middle, and a promise is not a middle.** 33.5 is roughly the *average* time for
6 km deliveries, so about half of them will be later than that. A promise that is missed half the time is
not a promise. 05-01's answer applies directly: if being late is worse than being early, promise a higher
quantile, not the fit.

**And the prediction has spread around it.** The residuals here have a standard deviation of about 1.6
minutes, so even at the same distance, times vary. A promise has to include a margin drawn from that
spread - and the size of the margin is a business decision about how often you are willing to be late,
which is exactly the pinball `q` from 05-01.

The honest handover: *"6 km deliveries average about 33 or 34 minutes, with a typical spread of a couple of
minutes either way. If you want to be right nine times out of ten, promise 36."*

## Debugging

### E15

**They almost certainly regressed distance on minutes instead of minutes on distance.**

The tell is the number itself: `1 / 3.5` = **0.2857**, and they report 0.29. A slope that is close to the
reciprocal of the expected one is the signature of swapped axes - the reversed fit gives 0.279 km per
minute here, which rounds to 0.28-0.29.

**The one-line check:** look at the units. A slope of 0.29 in a model of minutes must be *minutes per km*,
and 0.29 minutes per km means a 9 km delivery takes 2.6 minutes longer than a 1 km one. That is
transparently wrong, and it takes five seconds. **Checking a coefficient's units against common sense
catches more bugs than checking its value.**

(The second candidate, if the units are right: the feature is in metres rather than kilometres - but that
would give 0.0035, not 0.29.)

### E16

**Explanation 1: the line was not fitted by least squares.** A line fitted to absolute error, fitted by
eye, taken from a previous dataset, or produced by a model with regularisation (05-09) has no obligation
to make the residuals sum to zero.

**Explanation 2: the residuals are being computed against the wrong thing** - the wrong column, the wrong
subset of rows, or predictions from a model fitted on *different* rows. The classic version is computing
residuals on the test set: **a least-squares fit only guarantees residuals summing to zero on the rows it
was fitted to.** On held-out rows they can sum to anything, and a large sum there is a genuinely useful
signal that the two sets differ.

A third possibility worth ruling out: the model has no intercept. `LinearRegression(fit_intercept=False)`
removes the very equation that forces the sum to zero.

## Exam and interview reasoning

### E17

> "I want the line that minimises the sum of squared vertical distances. Write the error as a function of
> the intercept and the slope - it is a sum of squared terms, so it is a quadratic bowl in those two
> parameters, and the minimum is where both partial derivatives are zero. Differentiating with respect to
> the intercept gives 'the residuals sum to zero', which says the line passes through the mean of x and the
> mean of y. Differentiating with respect to the slope gives 'the residuals times x sum to zero', which
> says no linear relationship is left over. Solving those two together gives slope equal to the sum of the
> products of the deviations divided by the sum of the squared x-deviations. Because it is a quadratic
> bowl there is exactly one solution, which is why there is a formula at all."

**"What changes if I use absolute error instead?"**

> "The formula disappears. Absolute error is piecewise linear rather than quadratic, so the surface has
> kinks rather than a smooth bottom and the derivative does not exist everywhere - there is no closed form,
> and you solve it numerically or as a linear program. The fitted line also changes: it stops chasing
> outliers. On the nine deliveries I was working with, corrupting one point moved the least-squares slope
> from 3.5 to 6.8 and moved the absolute-error slope not at all."

## Transfer to a different situation

### E18

**The intercept is the predicted length of stay for a patient aged zero** - a newborn - and the data
contains nobody under 45.

**I would not report it.** It is an extrapolation of nearly half a century beyond the data, and the
relationship between age and length of stay over 45-92 has no reason to continue linearly down to zero.
Reporting it invites somebody to read it as "the baseline stay", which it is not.

**To make it interpretable: centre the age column** - replace `age` with `age - 65` (or `age - mean age`).
The slope is completely unchanged, and the intercept becomes **the predicted stay for a 65-year-old**,
which is a real patient and a number a clinician can sanity-check.

That is the general fix, and it costs nothing: **an intercept is only meaningful when zero is a value the
feature actually takes**, so move the zero to somewhere useful. It also has a second benefit that returns
in 05-03 and 05-09 - centring reduces the correlation between the intercept and the slope, which is the
tilt in this chapter's contour plot.

## Explain it to someone non-technical

### E19

> "Imagine plotting each delivery as a dot - distance across, time up - and then laying a ruler across the
> cloud. Every dot is some distance above or below the ruler, and that gap is how wrong the ruler would
> have been for that delivery. 'Best fit' means the position of the ruler that makes those gaps as small as
> possible overall. There is exactly one such position, and there is a formula for it, so nobody has to
> nudge the ruler around by eye."

(85 words.)

## Optional challenge

### E20 - is the surface really a bowl?

In [ ]:
def surface_error(intercept_value, slope_value):
    return float(np.mean((minutes - (intercept_value + slope_value * distance)) ** 2))


convexity_rng = np.random.default_rng(1)
violations, gaps = 0, []
for _ in range(200):
    first = (convexity_rng.uniform(-20, 40), convexity_rng.uniform(-5, 12))
    second = (convexity_rng.uniform(-20, 40), convexity_rng.uniform(-5, 12))
    midpoint = ((first[0] + second[0]) / 2, (first[1] + second[1]) / 2)

    at_midpoint = surface_error(*midpoint)
    average_of_ends = (surface_error(*first) + surface_error(*second)) / 2
    gaps.append(average_of_ends - at_midpoint)
    if at_midpoint > average_of_ends + 1e-9:
        violations += 1

print("200 random pairs of (intercept, slope):")
print("  the midpoint was never worse than the average of the ends in %d of them" % (200 - violations))
print("  violations: %d" % violations)
print("  the gap (average of ends minus midpoint) ranged from %.4f to %.1f"
      % (min(gaps), max(gaps)))
print("  and was never negative:", bool(min(gaps) >= -1e-9))

**No violations in 200 pairs, and the gap is never negative.** That is the definition of a **convex**
function: the surface never bulges above the straight line joining any two points on it.

Convexity is what makes least squares easy, and it is worth being explicit about what it buys:

- **There is exactly one minimum**, so "the best line" is well defined rather than being one of several
  candidates.
- **Any downhill path reaches it.** 03-08's gradient descent cannot get stuck, because there is nowhere to
  get stuck - every local minimum is the global one.
- **A closed form exists**, because setting the derivatives to zero gives linear equations rather than
  something that must be searched.

**What it would mean if the test ever failed**: the surface would have more than one basin, so gradient
descent's answer would depend on where it started, two people running the same code with different
initialisations could report different models, and "the best fit" would need qualifying with "found from
this starting point".

That is not a hypothetical - **it is the normal situation for neural networks** (module 10), and it is why
their results depend on a seed in a way linear regression's never do. Enjoy the bowl while it lasts.